In [1]:
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN,Embedding
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import imdb
import tensorflow as tf
import numpy as np

In [2]:
# Disable GPU
tf.config.set_visible_devices([], 'GPU')

In [3]:
# Set Vocabulary, Feature Dimension Size and Maximum Sentence Length

voc_size=10000
dim=128
max_length=500

In [4]:
# Load Data:
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=voc_size)
print(f"Traing data size: {X_train.shape}, Training label size: {y_train.shape}")
print(f"Testing data size: {X_test.shape}, Testing label size: {y_test.shape}")

Traing data size: (25000,), Training label size: (25000,)
Testing data size: (25000,), Testing label size: (25000,)


In [5]:
# Padding:
X_train=sequence.pad_sequences(X_train, maxlen=max_length)
X_test=sequence.pad_sequences(X_test, maxlen=max_length)
print(X_train[0])

[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [6]:
# Create Model:
model=Sequential()

In [7]:
model.add(Embedding(voc_size, dim, input_length=max_length))
model.add(SimpleRNN(dim, activation='relu'))               # 'tanh' is default activation function of SimpleRNN
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
# EarlyStopping:
early_stopping=EarlyStopping(monitor='val_loss',patience=5, restore_best_weights=True)

# Compile Model:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='acu')
])

In [9]:
# Train Model
history=model.fit(
    X_train,y_train,epochs=10, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping]
)


Epoch 1/10
625/625 [==============================] - 64s 102ms/step - loss: 7177643.5000 - accuracy: 0.6101 - precision: 0.6931 - recall: 0.3995 - acu: 0.6953 - val_loss: 0.5634 - val_accuracy: 0.7040 - val_precision: 0.7385 - val_recall: 0.6201 - val_acu: 0.7881
Epoch 2/10
625/625 [==============================] - 65s 104ms/step - loss: 0.4355 - accuracy: 0.7987 - precision: 0.8298 - recall: 0.7531 - acu: 0.8843 - val_loss: 0.4351 - val_accuracy: 0.8056 - val_precision: 0.7911 - val_recall: 0.8238 - val_acu: 0.8836
Epoch 3/10
625/625 [==============================] - 66s 105ms/step - loss: 0.2958 - accuracy: 0.8768 - precision: 0.8879 - recall: 0.8634 - acu: 0.9483 - val_loss: 0.4630 - val_accuracy: 0.7794 - val_precision: 0.8071 - val_recall: 0.7270 - val_acu: 0.8692
Epoch 4/10
625/625 [==============================] - 66s 106ms/step - loss: 0.2278 - accuracy: 0.9104 - precision: 0.9160 - recall: 0.9043 - acu: 0.9688 - val_loss: 0.4702 - val_accuracy: 0.8168 - val_precision: 0.79

In [10]:
results = model.evaluate(X_test, y_test, verbose=0)
for metric_name, metric_value in zip(model.metrics_names, results):
    print(f"{metric_name}: {metric_value:.4f}")

loss: 0.4486
accuracy: 0.7968
precision: 0.7951
recall: 0.7998
acu: 0.8754


In [11]:
model.save('simpleRNN_imdb.keras')